# PARTIE A — Chargement & choix des variables

Partie 1 Charger le fichier excel et le convertir en cvs

In [6]:
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')

In [7]:
df = pd.read_excel('../data/dataset_assurance_ML.xlsx')
df.to_csv('../data/dataset_assurance_ML.csv', index=False, encoding='utf-8-sig')
df = pd.read_csv('../data/dataset_assurance_ML.csv', encoding='utf-8-sig')
if df.empty:
    print ("Date set vide ")
else:
    print("Donnée bien chargées")

Donnée bien chargées


In [8]:
df.shape
df.head()

,N° Police,Nom,Prénom,Sexe,Âge,Catégorie Prof.,Salaire Annuel (€),Ville,Code Postal,Type Contrat,...,Type Véhicule,Usage Véhicule,Puissance Fiscale (CV),Valeur Véhicule (€),Coeff. Bonus-Malus,Nb Sinistres (3 ans),Montant Sinistres (€),Dernier Sinistre,Score Risque (0-100),Résiliation
0,ASS01000,Bernard,Valérie,F,56,Profession libérale,76288,Nice,6200,Gold,...,SUV,Loisirs,9,20108,0.80,1,1447,Dégât des eaux,12,0
1,ASS01001,Simon,Éric,H,69,Employé,30020,Paris,75018,Bronze,...,SUV,Loisirs,9,29845,1.02,1,685,Incendie,20,0
2,ASS01002,Lambert,Inès,F,46,Technicien,36995,Lyon,69009,Silver,...,Citadine,Domicile-Travail,4,19035,0.54,0,0,Aucun,0,0
3,ASS01003,Blanc,François,H,32,Profession libérale,64773,Bordeaux,33200,Gold,...,Berline,Loisirs,8,25765,0.90,1,0,Vol,28,1
4,ASS01004,Vincent,Patrick,H,60,Cadre,69870,Nantes,44200,Gold,...,Berline,Loisirs,7,24365,0.96,1,0,Incendie,7,0


In [9]:
TARGET = 'Résiliation'
print(df[TARGET].value_counts())

print ('en ourcentage') 
pourcentages = df[TARGET].value_counts(normalize=True) * 100
pourcentages.round(2)

Résiliation
0    450
1     50
Name: count, dtype: int64
en ourcentage


Résiliation
0    90.0
1    10.0
Name: proportion, dtype: float64

**Non**, le dataset est déséquilibré. La résiliation est la classe minoritaire.
consequence: le modèle cherchera à maximiser l'Accuracy globale en prédisant presque toujours 0

Etape 3: Détecter une fuite de données (data leakage)

In [10]:
print(pd.crosstab(df['Statut Contrat'], df[TARGET]))

Résiliation       0   1
Statut Contrat         
Actif           450   0
Résilié           0  36
Suspendu          0  14


je remarque avec la variable `status contrat`  nous avons un `100%` en actif et `100%` en resilié 
saut le dernier cas suspendu qui pose un proble 

`Nom` nous ne pouvons pas utilser  cette varible car déjà `etiquété` 

In [11]:
num_cols = [
    'Âge', 'Salaire Annuel (€)', 
    'Prime Annuelle (€)', 
    'Ancienneté (mois)',
    'Coeff. Bonus-Malus', 
    'Nb Sinistres (3 ans)',
    'Montant Sinistres (€)', 
    'Score Risque (0-100)'
    ]
cat_cols = ['Type Contrat', 
            'Catégorie Prof.', 
            'Usage Véhicule', 
            'Dernier Sinistre'
            ]

X = df[num_cols + cat_cols]
y = df[TARGET]
print(X.shape, y.shape)

(500, 12) (500,)


Etape 5:

En Machine Learning, calculer la corrélation d'une variable numérique avec la variable cible (Résiliation ou TARGET) permet de mesurer l'intensité du lien linéaire entre cette variable et le fait de résilier.
    - \Une corrélation proche de $+1$ : Quand la variable augmente, le risque de résiliation augmente fortement.
    - \Une corrélation proche de $-1$ : Quand la variable augmente, le risque de résiliation diminue fortement.
    - \Une corrélation proche de $0$ : La variable n'a pas de lien linéaire évident avec la résiliation.

In [12]:
print(X[num_cols].corrwith(y).round(3).sort_values(ascending=False))
print("#############################""")
print(df.groupby('Dernier Sinistre')[TARGET].mean().round(2).sort_values())

Nb Sinistres (3 ans)     0.455
Score Risque (0-100)     0.441
Coeff. Bonus-Malus       0.412
Montant Sinistres (€)    0.284
Prime Annuelle (€)       0.041
Âge                     -0.006
Salaire Annuel (€)      -0.012
Ancienneté (mois)       -0.055
dtype: float64
#############################
Dernier Sinistre
Aucun                    0.03
Incendie                 0.16
Catastrophe naturelle    0.18
Accident                 0.26
Bris de glace            0.30
Vol                      0.30
Dégât des eaux           0.36
Name: Résiliation, dtype: float64


les trois variables sont: `'Résiliation', 'Statut Contrat', 'ID_Client'`

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.20, random_state=42, stratify= y)
print(X_train.shape, X_test.shape)
print(y_train.mean().round(2), y_test.mean().round(2))

(400, 12) (100, 12)
0.1 0.1


In [14]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
preprocessor = ColumnTransformer([
('num', StandardScaler(), num_cols),
('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
candidats = {
'Régression Logistique': LogisticRegression(
max_iter=1000, class_weight='balanced', random_state=42),
'Random Forest': RandomForestClassifier(
n_estimators=300, max_depth=4, min_samples_leaf=10,
class_weight='balanced', random_state=42),
}
pipelines = {nom: Pipeline([('prep', preprocessor), ('model', algo)])
for nom, algo in candidats.items()}

In [16]:
from sklearn.model_selection import cross_val_score
for nom, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    print(f'{nom:22s} AUC = {scores.mean():.3f} ± {scores.std():.3f}')

Régression Logistique  AUC = 0.745 ± 0.113
Random Forest          AUC = 0.801 ± 0.096


Performance globale modérée (AUC $\approx$ 0,69) :Un score AUC de 0,5 correspond à un choix aléatoire (jet de pièce), tandis qu'un score de 1,0 représente un modèle parfait. Avec environ 0,69, vos deux modèles apprennent un signal utile, mais leurs performances restent moyennes.


Avantage pour le Random Forest :Le Random Forest obtient une meilleure capacité de discrimination (AUC légèrement plus élevé à 0,696) et, surtout, une meilleure stabilité ($\pm$ 0,095 contre $\pm$ 0,133 pour la Régression Logistique). Une variance plus faible indique qu'il réagit de façon plus constante d'un pli de validation croisée à un autre.Variabilité importante ($\pm$ 0,095 à $\pm$ 0,133) :L'écart-type est relativement élevé. Cela s'explique généralement par la taille réduite de l'échantillon ou par le déséquilibre de la variable cible (environ 10 % de résiliations).

# Réponses aux questions d'évaluation

* **Q1. Un modèle qui prédirait toujours « reste » aurait 90 % d'accuracy. Est-il meilleur que le vôtre ?**
  > **Non.** il obtiendrait 0 % de rappel sur la classe "Résilié" et ne détecterait aucun client à risque, malgré une accuracy trompeuse.

* **Q2. Pour le service Fidélisation, quelle erreur coûte le plus cher ?**
   **Rater un client qui va partir (Faux Négatif)**

* **Q3. Faut-il plutôt baisser ou monter le seuil ?**
  > **Baisser le seuil**Cela permet de détecter plus de clients à risque et d'agir proactivement avant qu'ils ne résilient.

In [17]:
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
confusion_matrix, classification_report)
pipeline = pipelines['Random Forest']
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]
print('Accuracy :', round(accuracy_score(y_test, y_pred),3))
print('F1:', round(f1_score(y_test, y_pred),3))
print('ROC-AUC :', round(roc_auc_score(y_test, y_proba),3))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Reste', 'Résilie']))

Accuracy : 0.87
F1: 0.519
ROC-AUC : 0.853
[[80 10]
 [ 3  7]]
              precision    recall  f1-score   support

       Reste       0.96      0.89      0.92        90
     Résilie       0.41      0.70      0.52        10

    accuracy                           0.87       100
   macro avg       0.69      0.79      0.72       100
weighted avg       0.91      0.87      0.88       100



Etape 12 sauvegarde du pipe ligne

In [18]:
import joblib, os
joblib.dump(pipeline, '../models/pipeline_resiliation.pkl')
print(os.path.getsize('../models/pipeline_resiliation.pkl') / 1024, 'Ko')

493.736328125 Ko


Etape 13 Sauvegarde des eto Donée 

In [19]:
import json
meta = {
'modele': 'Random Forest',
'auc_test': round(float(roc_auc_score(y_test, y_proba)), 3),
'num_cols': num_cols,
'cat_cols': cat_cols,
'num_ranges': {c: {'min': float(X[c].min()), 'max': float(X[c].max()),
'median': float(X[c].median())} for c in num_cols},
'cat_values': {c: sorted(X[c].unique().tolist()) for c in cat_cols},
}
with open('../models/metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

In [20]:
modele = joblib.load('../models/pipeline_resiliation.pkl')
client = pd.DataFrame([{
'Âge': 34, 'Salaire Annuel (€)': 28000, 'Prime Annuelle (€)': 950,
'Ancienneté (mois)': 6, 'Coeff. Bonus-Malus': 1.25, 'Nb Sinistres (3 ans)': 3,
'Montant Sinistres (€)': 4200, 'Score Risque (0-100)': 72,
'Type Contrat': 'Bronze', 'Catégorie Prof.': 'Entrepreneur',
'Usage Véhicule': 'Professionnel', 'Dernier Sinistre': 'Vol',
}])
print('Classe :', modele.predict(client))
print('Proba :', modele.predict_proba(client)[0, 1].round(3))

Classe : [1]
Proba : 0.816


Etape 15 erreur classique 

In [21]:
try:
    modele.predict(client.drop(columns=['Score Risque (0-100)']))
except Exception as e:
    print('ERREUR :', e)

ERREUR : columns are missing: {'Score Risque (0-100)'}
